# Data Inspection and Diagnostics

This notebook traces the data for a single fund through the entire pipeline to find any discrepancies.

In [ ]:
import pandas as pd
from sqlalchemy import create_engine
from config import db_config
import requests

# The fund name we are investigating
TARGET_FUND_NAME = 'Axis Bluechip Fund - Direct Plan - Growth'

## Step 1: Inspect the Raw Source Data
**Goal:** Let's get the master list of all funds directly from the API to find the **exact** `schemeName` and `schemeCode` for our target fund. A tiny difference in the name can cause it to be missed.

In [ ]:
print("Fetching master fund list from API...")
all_funds_url = "https://api.mfapi.in/mf"
response = requests.get(all_funds_url)
all_funds_df = pd.DataFrame(response.json())

# Search for our target fund in the raw list
target_fund_info = all_funds_df[all_funds_df['schemeName'].str.contains('Axis Bluechip Fund - Direct Plan - Growth', case=False)]

print("--- Raw Fund Information from API Source ---")
if not target_fund_info.empty:
    print("Fund found at the source!")
    display(target_fund_info)
    # Store the exact code and name for the next steps
    exact_scheme_code = target_fund_info.iloc[0]['schemeCode']
    exact_scheme_name = target_fund_info.iloc[0]['schemeName']
else:
    print("🛑 CRITICAL: Fund not found in the master list from the API.")

## Step 2: Inspect the Data Loaded into the Database
**Goal:** Now let's check our SQL database. Did the data for this fund, using the **exact name** we found in Step 1, actually get saved correctly?

In [ ]:
if 'exact_scheme_name' in locals():
    print(f"Querying the database for: '{exact_scheme_name}'")
    engine = create_engine(db_config.url)
    query = f"SELECT * FROM nav_data WHERE scheme_name = '{exact_scheme_name}'"
    
    try:
        db_df = pd.read_sql(query, engine)
        print("--- Data Found in SQL Database ---")
        if not db_df.empty:
            print(f"Success! Found {len(db_df)} records in the database.")
            print("Sample data from DB:")
            display(db_df.head())
        else:
            print("🛑 CRITICAL: Fund data was NOT found in the database, even though it exists at the source.")
            print("This suggests a problem during the data extraction or loading phase in your pipeline.")
    except Exception as e:
        print(f"An error occurred while querying the database: {e}")
else:
    print("Skipping database check because fund was not found at the source.")

## Step 3: Replicate the Analysis Notebook's Loading Process
**Goal:** Finally, let's replicate exactly what the forecasting notebook does to see why it's getting 0 records.

In [ ]:
print("--- Simulating the Forecasting Notebook --- ")
engine = create_engine(db_config.url)
full_query = "SELECT * FROM nav_data"
full_df_from_db = pd.read_sql(full_query, engine)

# This is the exact filter that was failing
analysis_df = full_df_from_db[full_df_from_db['scheme_name'] == TARGET_FUND_NAME]

print(f"Attempting to filter for: '{TARGET_FUND_NAME}'")
print(f"Number of records found: {len(analysis_df)}")

if len(analysis_df) == 0 and 'exact_scheme_name' in locals() and TARGET_FUND_NAME != exact_scheme_name:
    print("\n--- DIAGNOSIS ---")
    print("The problem is a name mismatch!")
    print(f"You are filtering for: '{TARGET_FUND_NAME}'")
    print(f"But the name in the database is: '{exact_scheme_name}'")
    print("Please use the exact name from the database in your analysis notebooks.")